In [4]:
### set up the notebook
import matplotlib.pyplot as plt
import pandas as pd
import xarray as xr
import numpy as np
import dask as da
from scipy.interpolate import griddata

from IPython.core.display import display, HTML
display(HTML("<style>.container { width:90% !important; }</style>"))
np.set_printoptions(linewidth=100) 

plt.rcParams.update({'font.size': 14})

da.config.set(**{'array.slicing.split_large_chunks': True})

In [2]:
### read in SWOT data and buoy loctaions

# SWOT
swot = xr.open_mfdataset('/data/hendreya/SWOT/SWOT_data/SWOT_L2_subsets_PGC0/SWOT_L2_LR_SSH_Expert_006_PGC0_02_bst.nc')

# buoy locations
buoy_loc = pd.read_csv('../data_to_publish/buoy_locations_FSP_latlon.csv')

In [5]:
### loop through SWOT times and save time and ECMWF at each buoy

# save swot times for each cycle
swot_times = swot.time.mean(axis=1)

# create empty arrays for rad and mod values
ecmwf = np.zeros((9, len(swot_times)))

# create a 2d array of swot lat and lon for interpolation by flattening the 2d arrays
lon = swot.longitude.values.flatten()
lat = swot.latitude.values.flatten()
swot_points = np.array([lon, lat]).T

# create array of buoy lats and lons
buoy_lats = buoy_loc.lat.values
buoy_lons = buoy_loc.lon.values


for i in range(len(swot_times)):
    print(i)
    swot_day = swot.isel(cycle=i)
    
    ecmwf_day = swot_day.model_wet_tropo_cor.values.flatten()
    
    # interpolate to the buoy location
    interp_mod = griddata(swot_points, ecmwf_day, (buoy_lons, buoy_lats), method='linear')

    ecmwf[:,i] = interp_mod

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95


In [11]:
### Create a new dataset with the interpolated values

# ECMWF
ecmwf_at_buoys = xr.Dataset(data_vars={'sn40': ('time', ecmwf[0,:]),
                                        'sn20': ('time', ecmwf[1,:]),
                                        'sn06': ('time', ecmwf[2,:]),
                                        'ss05': ('time', ecmwf[3,:]),
                                        'ss20': ('time', ecmwf[4,:]),
                                        'ss30': ('time', ecmwf[5,:]),
                                        'ss40': ('time', ecmwf[6,:]),
                                        'swxt': ('time', ecmwf[7,:]),
                                        'sext': ('time', ecmwf[8,:])},
                             coords={'time': swot_times.values}, 
                             attrs={'info': 'Wet correction from the ECMWF operational analysis from the SWOT L2 PGC0 product interpolated to buoy locations. Take the negative of this correction to get wet path delay.'})
    

In [13]:
### Save the datasets

# ECMWF
ecmwf_at_buoys.to_netcdf('../data_to_publish/ECMWF_at_buoys_FSP.nc')